### A. Using the mathematical representations developed in Task 2, write a program in either Python or R to solve the optimization problem computationally.

The goal is to determine how many tons Amazon should ship on each available route so that all fulfillment center demand is met at the lowest possible total cost.

### A1. Solver Solution

Before building the model, I organized the provided Excel data into separate tables for hubs, focus cities, fulfillment centers, and route costs.

The PuLP model used the CBC optimization solver. The solver returned the following results:

- Solver status: Optimal
- Minimum total cost: 182,376.25
- Decision variables: 192
- Constraints: 73

In [6]:
import pandas as pd
from pulp import (LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value)


# reading data from the excel
file_path = "Task3.xlsx"

centers = pd.read_excel(file_path, sheet_name="Centers").set_index("center_id")
hubs = pd.read_excel(file_path, sheet_name="Hubs").set_index("hub_id")
focus = pd.read_excel(file_path, sheet_name="Focus Cities").set_index("focus_id")
cost = pd.read_excel(file_path, sheet_name="Cost", na_values=["N/A"]).set_index("destination_id")

# creating dictionaries for costs
hub_focus_cost = {
    (h, f): cost.at[f, h]
    for h in hubs.index
    for f in focus.index
    if pd.notna(cost.at[f, h])
}

hub_center_cost = {
    (h, c): cost.at[c, h]
    for h in hubs.index
    for c in centers.index
    if pd.notna(cost.at[c, h])
}

focus_center_cost = {
    (f, c): cost.at[c, f]
    for f in focus.index
    for c in centers.index
    if pd.notna(cost.at[c, f])
}

# Creating the model
model = LpProblem("Amazon_Transportation", LpMinimize)

x = LpVariable.dicts("hub_to_focus", hub_focus_cost, lowBound=0)
y = LpVariable.dicts("hub_to_center", hub_center_cost, lowBound=0)
z = LpVariable.dicts("focus_to_center", focus_center_cost, lowBound=0)

# Objective function
model += (
    lpSum(hub_focus_cost[h, f] * x[h, f] for h, f in hub_focus_cost)
    + lpSum(hub_center_cost[h, c] * y[h, c] for h, c in hub_center_cost)
    + lpSum(focus_center_cost[f, c] * z[f, c] for f, c in focus_center_cost)
)

for h in hubs.index:
    model += (
        lpSum(x[h, f] for f in focus.index if (h, f) in hub_focus_cost)
        + lpSum(y[h, c] for c in centers.index if (h, c) in hub_center_cost)
        <= hubs.at[h, "capacity"]
    )

for f in focus.index:
    inflow = lpSum(x[h, f] for h in hubs.index if (h, f) in hub_focus_cost)
    outflow = lpSum(z[f, c] for c in centers.index if (f, c) in focus_center_cost)

    model += inflow <= focus.at[f, "capacity"]
    model += outflow == inflow

for c in centers.index:
    model += (
        lpSum(y[h, c] for h in hubs.index if (h, c) in hub_center_cost)
        + lpSum(z[f, c] for f in focus.index if (f, c) in focus_center_cost)
        == centers.at[c, "demand"]
    )

status = model.solve()

print("Solver status:", LpStatus[status])
print("Minimum total cost:", value(model.objective))

for variable in model.variables():
    if variable.varValue and variable.varValue > 0:
        print(variable.name, variable.varValue)


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/rob/Documents/WGU Projects/operations-research-projects/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/0bd932116cc6476aa289f3708baf0926-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /tmp/0bd932116cc6476aa289f3708baf0926-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 78 COLUMNS
At line 659 RHS
At line 733 BOUNDS
At line 734 ENDATA
Problem MODEL has 73 rows, 192 columns and 388 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 59 (-14) rows, 173 (-19) columns and 348 (-40) elements
Perturbing problem by 0.001% of 1.6 - largest nonzero change 0.00010083972 ( 0.020167943%) - largest zero change 5.0652857e-05
0  Obj 108142.7 Primal inf 96868.101 (54) Dual inf 1.8999118 (2)
50  Obj 180858.51 Primal inf 1618.2 (13)
64  Obj 182382.08
Optimal - objective value 18